# TD 05 - Regression NLP (Deep Learning)

## Objectif du TD
Construire une pipeline DL simple pour predire une note numerique a partir d'un texte.

Probleme a resoudre: predire la note (1 a 10) d'un avis patient.

Pipeline vise:
1) tokenization + sequences,
2) modele neural de regression,
3) entrainement,
4) evaluation avec MAE, MSE, RMSE, R2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)

## Etape 1 - Dataset

Dataset impose: `Mouwiya/drug-reviews`.

A faire:
- charger le dataset,
- identifier colonne texte + colonne note,
- prendre un sous-ensemble.

In [ ]:
from datasets import load_dataset

# 1) Chargement du dataset
ds = load_dataset("Mouwiya/drug-reviews")
TEXT_COL = "review"
Y_COL = "rating"

# On limite la taille pour garder un TD rapide a executer
N = min(10000, len(ds["train"]))
X = list(ds["train"][TEXT_COL][:N])
y = np.array(ds["train"][Y_COL][:N], dtype=float)

print("Nombre d'exemples utilises:", N)
print("Note minimale:", min(y), " | Note maximale:", max(y))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Nombre d'exemples utilises: 10000
Note minimale: 1.0  | Note maximale: 10.0


## Etape 2 - Split train/validation

A faire:
- creer X et y,
- split train/validation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Train:", len(X_train), "| Validation:", len(X_val))

Train: 8000 | Validation: 2000


## Etape 3 - Tokenization / Vectorization Keras

A faire:
- creer un `TextVectorization`,
- adapter sur X_train.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

max_tokens = 20000
seq_len = 180

vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=seq_len,
    standardize="lower_and_strip_punctuation",
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))

# Conversion explicite vers tf.string pour eviter les erreurs de dtype (ex: <U..., str127968)
X_train_tf = tf.constant(list(X_train), dtype=tf.string)
X_val_tf = tf.constant(list(X_val), dtype=tf.string)
y_train_arr = np.asarray(y_train, dtype=np.float32)
y_val_arr = np.asarray(y_val, dtype=np.float32)

## Etape 4 - Modeles DL de regression

A faire:
- modele 1: Embedding + LSTM + Dense(1),
- modele 2 (optionnel): Embedding + GRU + Dense(1),
- entrainer quelques epochs.

In [ ]:
from tensorflow import keras

# 3) Deux modeles simples: LSTM et GRU
def build_model(kind="lstm"):
    inp = keras.Input(shape=(), dtype=tf.string)
    x = vectorizer(inp)
    x = layers.Embedding(vectorizer.vocabulary_size(), 64, mask_zero=True)(x)

    if kind == "lstm":
        x = layers.LSTM(64, dropout=0.3, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    elif kind == "gru":
        x = layers.GRU(64, dropout=0.3, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    else:
        raise ValueError("kind doit etre 'lstm' ou 'gru'")

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="linear")(x)

    model = keras.Model(inp, out)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

trained_models = {}
for kind in ["lstm", "gru"]:
    print("\n" + "=" * 80)
    print("Training:", kind.upper())
    print("=" * 80)

    model = build_model(kind)
    early_stopping_callback = keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=3, restore_best_weights=True
    )
    model.fit(
        X_train_tf,
        y_train_arr,
        validation_data=(X_val_tf, y_val_arr),
        epochs=20,
        batch_size=64,
        callbacks=[early_stopping_callback],
        verbose=1,
    )
    trained_models[kind] = model

print("\nTraining termine. Passer a la cellule d'evaluation.")


Training: LSTM
Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 29.6752 - mae: 4.5197 - val_loss: 10.8281 - val_mae: 2.9692
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 13.1419 - mae: 3.0815 - val_loss: 10.1946 - val_mae: 2.7252
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 12.1225 - mae: 2.9109 - val_loss: 7.8466 - val_mae: 2.2318
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 8.7696 - mae: 2.3812 - val_loss: 7.4137 - val_mae: 2.1665
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 7.1829 - mae: 2.1210 - val_loss: 7.4180 - val_mae: 2.2218
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 6.5508 - mae: 2.0136 - val_loss: 6.9875 - val_mae: 1.9987
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 5.4042 - mae: 1.8301 - val_loss: 7.1371 - val_mae: 2.0573
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 5.4174 - mae: 1.7914 - val_loss: 7.0972 - val_mae: 2.0131
Epoch 9/20
125/125 ━━━━━━━━

## Etape 5 - Evaluation

Metriques:
- MAE,
- MSE,
- RMSE,
- R2.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rows = []

for kind, model in trained_models.items():
    pred = model.predict(X_val_tf, verbose=0).reshape(-1)
    pred = np.clip(pred, 1.0, 10.0)

    mae = mean_absolute_error(y_val_arr, pred)
    mse = mean_squared_error(y_val_arr, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val_arr, pred)

    rows.append({
        "Modele": kind.upper(),
        "MAE": round(mae, 4),
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
    })

df = pd.DataFrame(rows).sort_values("MAE")
display(df)

,Modele,MAE,MSE,RMSE,R2
0,LSTM,1.9975,6.9663,2.6394,0.3414
1,GRU,2.0309,7.3608,2.7131,0.3041
